In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# Reference parameters
config_params = {
    # Bayesian smoothing parameters
    "alpha_orders": 8,
    "alpha_reviews": 8,
    "global_late_rate": 0.0781,
    "global_cancel_rate": 0.0046,
    "global_claim_rate": 0.1486,
    
    # Thresholds
    "late_th": 0.01,
    "cancel_th": 0.0109,
    "claim_th": 0.05,
    
    # Maximum penalty caps
    "late_cap": 20,
    "cancel_cap": 20,
    "claim_cap": 20,
    
    # Logistic growth rates (k)
    "k_late": 10,
    "k_cancel": 10,
    "k_claim": 10
}

def apply_bayesian_smoothing(df: DataFrame, params: dict) -> DataFrame:
    """Applies Bayesian smoothing to raw rates to handle low volume sellers."""
    return df.withColumn(
        "late_rate_smoothed",
        (F.col("late_shipments_60d") + params["alpha_orders"] * params["global_late_rate"]) / 
        (F.col("orders_60d") + params["alpha_orders"])
    ).withColumn(
        "cancel_rate_smoothed",
        (F.col("cancellations_60d") + params["alpha_orders"] * params["global_cancel_rate"]) / 
        (F.col("orders_60d") + params["alpha_orders"])
    ).withColumn(
        "claim_rate_smoothed",
        (F.col("claims_60d") + params["alpha_reviews"] * params["global_claim_rate"]) / 
        (F.col("reviews_60d") + params["alpha_reviews"])
    )

def calculate_penalties(df: DataFrame, params: dict) -> DataFrame:
    """Calculates linear, quadratic and logistic penalties using smoothed rates."""
    metrics = ["late", "cancel", "claim"]
    
    for metric in metrics:
        th = params[f"{metric}_th"]
        cap = params[f"{metric}_cap"]
        k = params[f"k_{metric}"]
        
        col_raw = f"{metric}_rate"
        col_smoothed = f"{metric}_rate_smoothed"
        
        # Ratio and Capping
        df = df.withColumn(f"ratio_{metric}", F.col(col_smoothed) / th) \
               .withColumn(f"ratio_{metric}_capped", F.least(F.col(f"ratio_{metric}"), F.lit(cap)))
        
        # Linear Penalty 
            # Penalties are triggered only when the raw rate exceeds the threshold,
            # ensuring consistency with the operational targets communicated to sellers.
        df = df.withColumn(
            f"{metric}_capped_linear_penalty",
            F.when(F.col(col_raw) > th, F.col(f"ratio_{metric}_capped") / cap).otherwise(0)
        )
        
        # Quadratic Penalty
        df = df.withColumn(f"{metric}_capped_quadratic_penalty", F.col(f"{metric}_capped_linear_penalty") ** 2)
        
        # Logistic Penalty (Normalization + Sigmoid)
        df = df.withColumn(
            f"{metric}_ratio_capped_normalized",
            (F.col(f"ratio_{metric}_capped") - 1) / (cap - 1)
        ).withColumn(
            f"{metric}_capped_logistic_penalty",
            F.when(
                F.col(f"{metric}_ratio_capped_normalized") > 0,
                1 / (1 + F.exp(-k * (F.col(f"{metric}_ratio_capped_normalized") - 0.5)))
            ).otherwise(0)
        )
        
    return df

def generate_seller_index(df: DataFrame) -> DataFrame:
    """Calculates final seller indexes using different penalty models."""
    penalty_types = ["linear", "quadratic", "logistic"]
    
    for p_type in penalty_types:
        weighted_penalty = (
            2 * F.col(f"late_capped_{p_type}_penalty") + 
            F.col(f"cancel_capped_{p_type}_penalty") + 
            F.col(f"claim_capped_{p_type}_penalty")
        )
        
        df = df.withColumn(
            f"seller_index_{p_type}",
            F.round(F.greatest(F.lit(1), 5 - weighted_penalty), 0)
        )
    return df


In [0]:
# --- MAIN EXECUTION ---
m_metrics = spark.table("marketplace_olist.gold.fact_seller_metrics")

# Using lambda to pass the params dict to the transform functions
gold_seller_index = (
    m_metrics
    .transform(lambda df: apply_bayesian_smoothing(df, config_params))
    .transform(lambda df: calculate_penalties(df, config_params))
    .transform(generate_seller_index)
)

In [0]:
# 3. Save to Gold Medallion Layer
gold_seller_index.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("marketplace_olist.gold.fact_seller_index")